# 电商销售数据分析 — Online Retail Dataset
**作者**: 周余钌 | **数据集**: UCI Online Retail（英国在线零售交易数据，2010-2011）
---

## 第 1 部分：导入依赖库

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import csv, io as sio, os, sys

plt.rcParams["font.sans-serif"] = ["SimHei", "Microsoft YaHei", "Arial Unicode MS", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
sns.set_theme(style="whitegrid", palette="Set2")

print("Libraries imported successfully!")
print(f"  pandas {pd.__version__} | numpy {np.__version__} | seaborn {sns.__version__}")

## 第 2 部分：数据加载

In [ ]:
# 项目根目录（自动定位，不依赖工作目录）
_proj_dir = os.path.dirname(os.path.abspath("online_retail_analysis.ipynb"))
os.chdir(_proj_dir)
if _proj_dir not in sys.path:
    sys.path.insert(0, _proj_dir)

file_path = os.path.join(_proj_dir, "Online Retail.xlsx")
print(f"加载路径: {file_path}")

# 读取魔术字节判断真实格式
with open(file_path, "rb") as f:
    magic = f.read(4)

if magic[:2] == b"PK":
    print("格式: Excel xlsx")
    df = pd.read_excel(file_path, sheet_name=0, engine="openpyxl", dtype=str, header=0)
elif magic[:2] == b"\xd0\xcf":
    print("格式: 旧版 Excel xls")
    df = pd.read_excel(file_path, sheet_name=0, engine="xlrd", dtype=str, header=0)
else:
    print("格式: CSV 文本文件")
    with open(file_path, "r", encoding="latin-1") as f:
        first_line = f.readline()
    sep = "\t" if first_line.count("\t") > first_line.count(",") else ","
    df = pd.read_csv(file_path, encoding="latin-1", sep=sep, dtype=str)

# ── 关键修复：若只有1列，说明数据是逗号拼接字符串，需要拆分 ──
if df.shape[1] == 1:
    print(f"检测到单列格式，自动拆分...")
    col_header = df.columns[0]
    col_names  = [c.strip() for c in col_header.split(",")]

    parsed_rows = []
    for val in df[col_header]:
        if pd.isna(val) or str(val).strip() == "":
            continue
        try:
            row = next(csv.reader(sio.StringIO(str(val))))
            if len(row) == len(col_names):
                parsed_rows.append(row)
            elif len(row) > len(col_names):
                # Description 字段含逗号时多出列，合并中间多余部分
                extra = len(row) - len(col_names)
                fixed = row[:2] + [",".join(row[2:2+extra+1])] + row[2+extra+1:]
                parsed_rows.append(fixed)
        except Exception:
            continue
    df = pd.DataFrame(parsed_rows, columns=col_names)
    print(f"拆分完成: {df.shape[0]:,} 行 × {df.shape[1]} 列")

print(f"\n加载成功: {df.shape[0]:,} 行 × {df.shape[1]} 列")
print(f"列名: {df.columns.tolist()}")
df.head(3)

## 第 3 部分：数据清洗

In [ ]:
print("清洗前缺失值:")
print(df.isnull().sum()[df.isnull().sum() > 0])

df_clean = df.copy()

# 1. 去重
before = len(df_clean)
df_clean.drop_duplicates(inplace=True)
print(f"\n去重: {before:,} -> {len(df_clean):,} (删除 {before-len(df_clean):,} 行)")

# 2. 缺失值填充
df_clean["CustomerID"] = (df_clean["CustomerID"].astype(str)
                          .replace("nan", "Unknown").replace("", "Unknown").fillna("Unknown"))
df_clean["Description"] = df_clean["Description"].fillna("Unknown Product")

# 3. 数值转换 + 过滤异常值
df_clean["Quantity"]  = pd.to_numeric(df_clean["Quantity"],  errors="coerce")
df_clean["UnitPrice"] = pd.to_numeric(df_clean["UnitPrice"], errors="coerce")
before = len(df_clean)
df_clean = df_clean[(df_clean["Quantity"] > 0) & (df_clean["UnitPrice"] > 0)]
print(f"过滤异常值: 删除 {before-len(df_clean):,} 行")

# 4. 退货订单
before = len(df_clean)
df_clean = df_clean[~df_clean["InvoiceNo"].astype(str).str.startswith("C")]
print(f"退货订单: 删除 {before-len(df_clean):,} 行")

# 5. 销售额
df_clean["AmountSpent"] = df_clean["Quantity"] * df_clean["UnitPrice"]

# 6. 日期（兼容 pandas 1.x / 2.x）
df_clean["InvoiceDate"] = pd.to_datetime(df_clean["InvoiceDate"])
df_clean["Year"]      = df_clean["InvoiceDate"].dt.year
df_clean["Month"]     = df_clean["InvoiceDate"].dt.month
df_clean["YearMonth"] = df_clean["InvoiceDate"].dt.to_period("M")
df_clean["DayOfWeek"] = df_clean["InvoiceDate"].dt.dayofweek
df_clean["Hour"]      = df_clean["InvoiceDate"].dt.hour

print(f"\n清洗完成: {df_clean.shape[0]:,} 行 × {df_clean.shape[1]} 列")
df_clean[["Quantity", "UnitPrice", "AmountSpent"]].describe().round(2)

## 第 4 部分：EDA
### 4.1 整体销售概况

In [ ]:
valid = df_clean[df_clean["CustomerID"] != "Unknown"]

total_gmv       = df_clean["AmountSpent"].sum()
total_orders    = df_clean["InvoiceNo"].nunique()
total_customers = valid["CustomerID"].nunique()
avg_order_value = df_clean.groupby("InvoiceNo")["AmountSpent"].sum().mean()
avg_unit_price  = df_clean["UnitPrice"].mean()
repeat_rate     = (valid.groupby("CustomerID")["InvoiceNo"].nunique() > 1).mean() * 100

print("-" * 48)
print("           KEY BUSINESS METRICS")
print("-" * 48)
print(f"  Total GMV          : GBP {total_gmv:>12,.2f}")
print(f"  Total Orders       :     {total_orders:>12,}")
print(f"  Total Customers    :     {total_customers:>12,}")
print(f"  Avg Order Value    : GBP {avg_order_value:>12,.2f}")
print(f"  Avg Unit Price     : GBP {avg_unit_price:>12,.2f}")
print(f"  Repeat Purchase %  :     {repeat_rate:>11.1f}%")
print("-" * 48)

### 4.2 月度销售趋势

In [ ]:
os.makedirs("outputs", exist_ok=True)

monthly_sales = df_clean.groupby("YearMonth")["AmountSpent"].sum().reset_index()
monthly_sales["YearMonth_str"] = monthly_sales["YearMonth"].astype(str)
monthly_sales["MoM_Growth"]    = monthly_sales["AmountSpent"].pct_change() * 100

peak     = monthly_sales.loc[monthly_sales["AmountSpent"].idxmax()]
peak_idx = monthly_sales["AmountSpent"].idxmax()
print(f"Peak month: {peak['YearMonth']}  GBP {peak['AmountSpent']:,.2f}")

fig, ax = plt.subplots(figsize=(14, 5))
ax.fill_between(monthly_sales["YearMonth_str"], monthly_sales["AmountSpent"], alpha=0.15, color="#2E86AB")
ax.plot(monthly_sales["YearMonth_str"], monthly_sales["AmountSpent"],
        marker="o", linewidth=2.5, markersize=5, color="#2E86AB")
ax.annotate(f"Peak\nGBP {peak['AmountSpent']:,.0f}",
            xy=(peak_idx, peak["AmountSpent"]),
            xytext=(max(peak_idx-2, 0), peak["AmountSpent"]*1.08),
            arrowprops=dict(arrowstyle="->", color="#E63946", lw=1.5),
            fontsize=9, color="#E63946", ha="center")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"GBP {x/1e3:.0f}K"))
ax.set_title("Monthly Sales Trend (GMV)", fontsize=14, fontweight="bold")
ax.set_xlabel("Year-Month"); ax.set_ylabel("Sales Amount")
plt.xticks(rotation=45, ha="right", fontsize=9); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("outputs/monthly_sales_trend.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: outputs/monthly_sales_trend.png")

### 4.3 国家 / 地区销售分布

In [ ]:
country_sales = df_clean.groupby("Country")["AmountSpent"].sum().sort_values(ascending=False)
uk_pct = country_sales.get("United Kingdom", 0) / total_gmv * 100

print("Top 10 countries by revenue:")
for rank, (country, sales) in enumerate(country_sales.head(10).items(), 1):
    bar = "#" * int(sales / country_sales.iloc[0] * 28)
    print(f"  {rank:2}. {country:<25} GBP {sales:>10,.0f}  {sales/total_gmv*100:5.1f}%  {bar}")
print(f"\nUK market share: {uk_pct:.1f}%")

pie_data = pd.concat([country_sales.head(9),
                      pd.Series({"Others": country_sales.iloc[9:].sum()})])
colors = sns.color_palette("Set2", len(pie_data))
fig, ax = plt.subplots(figsize=(11, 7))
wedges, texts, autotexts = ax.pie(pie_data.values, labels=pie_data.index, autopct="%1.1f%%",
    colors=colors, startangle=140,
    wedgeprops={"linewidth": 0.8, "edgecolor": "white"}, textprops={"fontsize": 10})
for at in autotexts:
    at.set_color("white"); at.set_fontweight("bold"); at.set_fontsize(9)
ax.set_title("Sales Distribution by Country", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("outputs/country_sales_pie.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: outputs/country_sales_pie.png")

### 4.4 用户购买行为分析

In [ ]:
valid_cust   = df_clean[df_clean["CustomerID"] != "Unknown"]
purchase_cnt = valid_cust.groupby("CustomerID")["InvoiceNo"].nunique().reset_index(name="PurchaseCount")
spending     = valid_cust.groupby("CustomerID")["AmountSpent"].sum().reset_index(name="TotalSpent")

single = (purchase_cnt["PurchaseCount"] == 1).sum()
repeat = (purchase_cnt["PurchaseCount"] > 1).sum()
total  = len(purchase_cnt)
spending_sorted = spending.sort_values("TotalSpent", ascending=False)
top20     = spending_sorted.iloc[:int(len(spending_sorted) * 0.2)]
top20_pct = top20["TotalSpent"].sum() / spending_sorted["TotalSpent"].sum() * 100

print(f"Valid customers : {total:,}")
print(f"Repeat buyers   : {repeat:,}  ({repeat/total*100:.1f}%)")
print(f"One-time buyers : {single:,}  ({single/total*100:.1f}%)")
print(f"Top 20% customers => {top20_pct:.1f}% of revenue")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
freq_dist = purchase_cnt["PurchaseCount"].value_counts().sort_index().head(15)
axes[0].bar(freq_dist.index.astype(str), freq_dist.values, color="#5BC0BE", edgecolor="white")
axes[0].set_title("Customer Purchase Frequency (Top 15)", fontsize=12, fontweight="bold")
axes[0].set_xlabel("Number of Purchases"); axes[0].set_ylabel("Customer Count")
axes[0].tick_params(axis="x", rotation=45)
axes[1].pie([single, repeat], labels=["One-time", "Repeat"], autopct="%1.1f%%",
    colors=["#FC9E4F", "#2E86AB"], startangle=90,
    wedgeprops={"linewidth": 0.8, "edgecolor": "white"}, textprops={"fontsize": 11})
axes[1].set_title("One-time vs Repeat Customers", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig("outputs/customer_purchase_analysis.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: outputs/customer_purchase_analysis.png")

### 4.5 热销商品分析

In [ ]:
product_revenue = (df_clean.groupby(["StockCode", "Description"])["AmountSpent"]
                   .sum().reset_index().sort_values("AmountSpent", ascending=False))

print("Top 10 by Revenue:")
for i, (_, row) in enumerate(product_revenue.head(10).iterrows(), 1):
    print(f"  {i:2}. {str(row['Description'])[:45]:<45} GBP {row['AmountSpent']:>10,.2f}")

top10 = product_revenue.head(10).copy()
top10["ShortDesc"] = top10["Description"].str[:40]
fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(top10["ShortDesc"], top10["AmountSpent"], color="#2E86AB", edgecolor="white")
ax.invert_yaxis()
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"GBP {x/1e3:.0f}K"))
ax.set_xlabel("Sales Amount")
ax.set_title("Top 10 Products by Revenue", fontsize=14, fontweight="bold")
for bar, val in zip(bars, top10["AmountSpent"]):
    ax.text(bar.get_width() + total_gmv*0.001, bar.get_y() + bar.get_height()/2,
            f"GBP {val:,.0f}", va="center", fontsize=8.5)
plt.tight_layout()
plt.savefig("outputs/top10_products.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: outputs/top10_products.png")

## 第 5 部分：RFM 用户价值分层

In [ ]:
from analysis_functions import calculate_rfm, classify_users

rfm = calculate_rfm(df_clean)
rfm = classify_users(rfm)

order = ["高价值用户", "中价值用户", "低价值用户", "休眠用户"]
seg_summary = (rfm.groupby("UserSegment")
               .agg(Count=("CustomerID","count"), AvgR=("Recency","mean"),
                    AvgF=("Frequency","mean"), AvgM=("Monetary","mean")).round(1))
seg_summary = seg_summary.reindex([s for s in order if s in seg_summary.index])
print(seg_summary.to_string())

colors_rfm = ["#2E86AB", "#5BC0BE", "#FC9E4F", "#C9CBA3"]
fig, axes  = plt.subplots(1, 2, figsize=(13, 5))
seg_counts = rfm["UserSegment"].value_counts().reindex(
    [s for s in order if s in rfm["UserSegment"].unique()])
seg_mon    = rfm.groupby("UserSegment")["Monetary"].mean().reindex(
    [s for s in order if s in rfm["UserSegment"].unique()])
axes[0].bar(seg_counts.index, seg_counts.values, color=colors_rfm[:len(seg_counts)], edgecolor="white")
axes[0].set_title("Customers per RFM Segment", fontsize=12, fontweight="bold")
axes[0].tick_params(axis="x", rotation=15)
axes[1].bar(seg_mon.index, seg_mon.values, color=colors_rfm[:len(seg_mon)], edgecolor="white")
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"GBP {x:,.0f}"))
axes[1].set_title("Avg Spend per RFM Segment", fontsize=12, fontweight="bold")
axes[1].tick_params(axis="x", rotation=15)
plt.tight_layout()
plt.savefig("outputs/rfm_segments.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: outputs/rfm_segments.png  ({len(rfm):,} customers segmented)")

## 第 6 部分：数据洞察与业务建议

In [ ]:
top5_pct      = country_sales.head(5).sum() / total_gmv * 100
top10_rev_pct = product_revenue.head(10)["AmountSpent"].sum() / total_gmv * 100
top20_rev_pct = top20["TotalSpent"].sum() / spending_sorted["TotalSpent"].sum() * 100

print("=" * 58)
print("        DATA INSIGHTS & BUSINESS RECOMMENDATIONS")
print("=" * 58)
print(f"  Total GMV        : GBP {total_gmv:>12,.2f}")
print(f"  Total Orders     :     {total_orders:>12,}")
print(f"  Total Customers  :     {total_customers:>12,}")
print(f"  Avg Order Value  : GBP {avg_order_value:>12,.2f}")
print(f"  Repeat Rate      :     {repeat_rate:>11.1f}%")
print("-" * 58)
print(f"  1. UK = {uk_pct:.1f}%  |  Top-5 countries = {top5_pct:.1f}%")
print(f"     Risk: over-reliance on single market")
print(f"  2. Repeat rate {repeat_rate:.1f}%  |  Top 20% customers => {top20_rev_pct:.1f}% revenue")
print(f"     Opportunity: loyalty programme")
print(f"  3. Top-10 SKUs => {top10_rev_pct:.1f}% revenue")
print(f"     Action: ensure top SKU stock availability")
print("-" * 58)
print("  [+] Membership rewards   => est. +5-10% repeat rate")
print("  [+] Free-ship DE/FR test => est. +15% intl revenue")
print("  [+] Tail SKU clearance   => cut storage cost")
print("=" * 58)

## 第 7 部分：导出分析结果

In [ ]:
os.makedirs("outputs", exist_ok=True)

monthly_sales.drop(columns=["YearMonth"]).rename(columns={"YearMonth_str": "YearMonth"})     .to_csv("outputs/monthly_sales.csv", index=False, encoding="utf-8-sig")

cs = country_sales.reset_index()
cs.columns = ["Country", "TotalSales"]
cs["Pct%"] = (cs["TotalSales"] / total_gmv * 100).round(2)
cs.to_csv("outputs/country_sales.csv", index=False, encoding="utf-8-sig")

product_revenue.to_csv("outputs/product_sales_ranking.csv", index=False, encoding="utf-8-sig")

purchase_cnt.merge(spending, on="CustomerID", how="left")     .to_csv("outputs/customer_summary.csv", index=False, encoding="utf-8-sig")

rfm.to_csv("outputs/rfm_results.csv", index=False, encoding="utf-8-sig")

print("All outputs saved to outputs/")
for f in ["monthly_sales.csv","country_sales.csv","product_sales_ranking.csv",
          "customer_summary.csv","rfm_results.csv"]:
    size = os.path.getsize(f"outputs/{f}")
    print(f"  {f:<35} {size/1024:.1f} KB")